# nockasm tour

A thin macro layer over canonical Nock 4K. Five primitives, deterministic lowering, output runs on `pinochle.nock`. This notebook walks through each macro and shows what it expands to.

Run with a regular Python kernel. Then copy the expanded canonical Nock into a `Nock 4K` kernel notebook to evaluate interactively.

In [ ]:
from nockasm import expand
from pinochle import nock, parse_noun

def show(src, subject=None):
    """Print assembly source, canonical Nock, and (if subject given) the
    pinochle evaluation result. This is the pedagogical view."""
    canonical = expand(src)
    pretty    = expand(src, pretty=True)
    print('asm:'); print('  ' + src.strip().replace('\n', '\n  '))
    print('canonical:'); print('  ' + canonical)
    print('pretty:');    print('  ' + pretty)
    if subject is not None:
        s = parse_noun(subject)
        f = parse_noun(canonical)
        print(f'subject:  {subject}')
        print(f'result:   {nock(s, f)}')

## 1. Named opcodes

The cheapest win. `%inc` instead of `4`. `%slot` instead of `0`. Reading code where opcodes are named is unambiguous.

In [ ]:
show('(%inc (%slot 1))', subject='41')

In [ ]:
show('(%eq (%slot 2) (%slot 3))', subject='[42 42]')

The full opcode list:

| Asm | Nock | Meaning |
|---|---|---|
| `(%slot N)`    | `[0 N]`     | tree axis N from subject |
| `(%const X)`   | `[1 X]`     | quote literal X |
| `(%eval S F)`  | `[2 S F]`   | evaluate F against result of S |
| `(%isa F)`     | `[3 F]`     | is F a cell? |
| `(%inc F)`     | `[4 F]`     | increment |
| `(%eq F G)`    | `[5 F G]`   | equality (0=yes, 1=no) |
| `(%if C T E)`  | `[6 C T E]` | branch |
| `(%comp F G)`  | `[7 F G]`   | F then G |
| `(%push F G)`  | `[8 F G]`   | push F's value onto subject, then G |
| `(%call N F)`  | `[9 N F]`   | invoke arm N of core produced by F |
| `(%edit N V F)`| `[10 [N V] F]` | replace axis N with V's result, then F |
| `(%hint T F)`  | `[11 T F]`  | static hint |
| `(%hintd T C F)` | `[11 [T C] F]` | dynamic hint (clue is a formula) |

## 2. Axis schemas

Declare the shape of the subject. Names resolve to axes.

Flat schemas are right-leaning, matching Hoon's convention for lists:

```
{.a .b .c}   ; .a=2, .b=6, .c=7
```

Explicit nesting overrides:

```
{{.a .b} .c}   ; .a=4, .b=5, .c=3
```

In [ ]:
show("""
:subject {.a .b .c}
(%inc .b)
""", subject='[10 20 30]')

## 3. `#let`

Push a value onto the subject via opcode 8. The new name lives at axis 2 of the augmented subject. Existing names are shifted rightward via `+peg(3, n)` so they still resolve in the body.

In [ ]:
show("""
:subject {.x .y}
#let .doubled = (%inc (%inc .x)) in
  (%eq .doubled .y)
""", subject='[3 5]')

# .x=2, .y=3.
# value: [4 4 0 2]  (inc inc slot 2)
# body axes: .doubled=2, .x=peg(3,2)=6, .y=peg(3,3)=7
# body: [5 [0 2] [0 7]]
# full: [8 [4 4 0 2] [5 [0 2] [0 7]]]
# against [3 5]: doubled=5, eq(5,5)=0 (yes)

In [ ]:
# Bare atom values get lifted to [1 atom] automatically.
show("""
:subject .x
#let .v = 42 in (%eq .v .x)
""", subject='42')

## 4. `#match`

Head-tag dispatch. The scrutinee is evaluated once (via opcode 8) and compared to each literal pattern in turn. The `_ =>` default is required.

Patterns are literal nouns. Arm bodies are formulas (bare atoms lift).

In [ ]:
src = """
:subject {.tag .data}
#match .tag {
  1 => (%inc .data)
  2 => .data
  _ => 0
}
"""
show(src, subject='[1 41]')
print()
show(src, subject='[2 41]')
print()
show(src, subject='[99 41]')

## 5. Worked example: edit middle of a triple

All four features at once.

In [ ]:
show("""
:subject {.before .target .after}
#let .next = (%inc .target) in
  [.before .next .after]
""", subject='[10 41 99]')

## Workflow with the Nock kernel

Until a `:asm` cell magic exists, the pattern is:

1. Write the assembly here in a Python cell.
2. `expand()` it.
3. Copy the canonical Nock into a `:formula` cell in a Nock 4K kernel notebook.

Example output to paste — set `:subject [10 41 99]`, then `:formula`:

In [ ]:
print(expand("""
:subject {.before .target .after}
#let .next = (%inc .target) in
  [.before .next .after]
"""))

## Escape hatch: raw Nock when you need it

`[a b c ...]` cells aren't macro-expanded. Useful for the cons-formula distribution pattern: a cell-headed formula produces a cell by evaluating each part.

In [ ]:
# Build a cell of [inc(a) inc(b)] via Nock's distribution rule:
#   *[s [b c] d] = [*[s [b c]] *[s d]]
show("""
:subject {.a .b}
[(%inc .a) (%inc .b)]
""", subject='[3 5]')